# Capstone — mirrors your deployed research paper

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ubaidrees/flyrank-ml-tasks/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

Research question: Can search-performance signals (impressions, position, CTR, freshness) predict which content will decline in visibility, well enough to prioritize a content team's limited review capacity? Decision supported: which pages a content team should review first each cycle, and whether a trained model earns its complexity over a simple, transparent rule.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

FlyRank ML Internship starter release (content_refresh_anonymized.csv), filtered to active content (impressions_90d > 0, content_age_days >= 90), de-duplicated by content_id. Single historical snapshot. Excludes content under 90 days old (insufficient trend history) and any row-level private identifiers — all IDs are anonymized hashes.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("https://raw.githubusercontent.com/Ubaidrees/flyrank-ml-tasks/main/data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)
df = df[(df['impressions_90d'] > 0) & (df['content_age_days'] >= 90)].copy()
df = df.drop_duplicates(subset='content_id')

# Fix 1: CTR unit-scale bug (1,689 rows stored as 0-100 instead of 0-1) — recompute from raw counts
df['ctr'] = (df['clicks_90d'] / df['impressions_90d']).clip(upper=1.0)

print(f"Rows after filtering: {len(df):,}")
print(f"Positive rate (declining): {df['is_declining_label'].mean():.3f}")

Rows after filtering: 30,000
Positive rate (declining): 0.542


## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

Label: is_declining_label = 1 if trend_direction == "down". Twelve features (see paper). Baseline rule stale_page_ctr_gap, both signals independently verified before use. Validation: client-grouped 80/20 split, zero client overlap. Leakage check: impressions_90d partially overlaps the label's own window.

In [2]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

feature_cols = [c for c in [
    'impressions_90d', 'clicks_90d', 'sessions_90d', 'ai_sessions_90d',
    'content_age_days', 'days_since_last_update', 'ctr', 'avg_position',
    'engagement_rate', 'scroll_rate', 'word_count', 'ai_traffic_pct'
] if c in df.columns]

# Fix 2: flag bulk-update artifact clients (55.8% of rows)
bulk_flag = df.groupby('client_id')['days_since_last_update'].transform(
    lambda x: x.value_counts(normalize=True).max() > 0.5
)
df['bulk_event_flag'] = bulk_flag

X = df[feature_cols].fillna(0)
y = df['is_declining_label']
groups = df['client_id']

gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

overlap = set(df.iloc[train_idx]['client_id']) & set(df.iloc[test_idx]['client_id'])
print(f"Client overlap (honest split): {len(overlap)} (must be 0)")
print(f"Bulk-event rows: {df['bulk_event_flag'].sum():,} of {len(df):,} ({df['bulk_event_flag'].mean()*100:.1f}%)")

Client overlap (honest split): 0 (must be 0)
Bulk-event rows: 16,740 of 30,000 (55.8%)


## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

Client-grouped AUC vs. naive-split AUC vs. baseline rule, on the same data.

In [3]:
# Baseline rule, full dataset
baseline_flag = (
    (df['days_since_last_update'] >= 180) & (df['impressions_90d'] >= 500) &
    (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)
)
print(f"Baseline flagged: {baseline_flag.sum()} of {len(df)}")

# Honest model
model = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced')
model.fit(X_train, y_train)
model_proba = model.predict_proba(X_test)[:, 1]
honest_auc = roc_auc_score(y_test, model_proba)

# Naive split, for comparison
X_tr_n, X_te_n, y_tr_n, y_te_n, idx_tr_n, idx_te_n = train_test_split(X, y, df.index, test_size=0.2, random_state=42, stratify=y)
model_naive = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced')
model_naive.fit(X_tr_n, y_tr_n)
naive_auc = roc_auc_score(y_te_n, model_naive.predict_proba(X_te_n)[:, 1])
naive_overlap = set(df.loc[idx_tr_n, 'client_id']) & set(df.loc[idx_te_n, 'client_id'])

print(f"Honest (client-grouped) AUC: {honest_auc:.3f}")
print(f"Naive (random split) AUC: {naive_auc:.3f}  |  client overlap: {len(naive_overlap)} clients")

# Leakage check
no_imp = [c for c in feature_cols if c != 'impressions_90d']
model_no_imp = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=42, class_weight='balanced')
model_no_imp.fit(X_train[no_imp], y_train)
auc_no_imp = roc_auc_score(y_test, model_no_imp.predict_proba(X_test[no_imp])[:, 1])
print(f"AUC without impressions_90d: {auc_no_imp:.3f}")

# Error breakdown
y_pred = (model_proba >= 0.5).astype(int)
fn = ((y_test == 1) & (y_pred == 0)).sum()
fp = ((y_test == 0) & (y_pred == 1)).sum()
print(f"False negatives: {fn}  |  False positives: {fp}")

Baseline flagged: 10 of 30000
Honest (client-grouped) AUC: 0.605
Naive (random split) AUC: 0.745  |  client overlap: 31 clients
AUC without impressions_90d: 0.551
False negatives: 1269  |  False positives: 1350


## 5. Limitations

*What this work cannot claim.*

Single snapshot; bulk-update artifact (55.8% of rows); CTR unit bug (fixed); non-monotonic staleness; partial impressions_90d leakage (0.053 AUC); not validated on new clients.

In [4]:
# Reproduce the bulk-event decline-rate gap cited in the paper
for flag in [True, False]:
    subset = df[df['bulk_event_flag'] == flag]
    print(f"bulk_event_flag={flag}: n={len(subset)}, decline rate={subset['is_declining_label'].mean():.3f}")

bulk_event_flag=True: n=16740, decline rate=0.563
bulk_event_flag=False: n=13260, decline rate=0.516


## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

Reason codes and actions, per the playbook built in Week 7.

In [5]:
df['risk_score'] = model.predict(X) if False else model.predict_proba(X)[:, 1]

def assign_reason(row):
    if row['days_since_last_update'] >= 180 and row['impressions_90d'] >= 500 and 0 < row['avg_position'] <= 20 and row['ctr'] < 0.5:
        return 'stale_page_ctr_gap'
    elif row['impressions_90d'] >= 5000 and row['risk_score'] >= 0.5:
        return 'high_volume_risk'
    elif row['days_since_last_update'] >= 180:
        return 'stale_needs_review'
    elif row['ctr'] < 0.15 and 0 < row['avg_position'] <= 20:
        return 'low_ctr_visible_page'
    else:
        return 'general_monitor'

df['reason_code'] = df.apply(assign_reason, axis=1)
print(df['reason_code'].value_counts())

reason_code
low_ctr_visible_page    17851
general_monitor          8783
high_volume_risk         3195
stale_needs_review        161
stale_page_ctr_gap         10
Name: count, dtype: int64


## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

Confirms the same figure and metrics file the deployed paper references are present and current.

In [6]:
import os, json
os.makedirs("work/outputs", exist_ok=True)
os.makedirs("work/figures", exist_ok=True)

metrics = {
    "honest_auc_client_grouped": round(honest_auc, 3),
    "naive_auc_ungrouped_DO_NOT_QUOTE": round(naive_auc, 3),
    "auc_without_impressions_90d": round(auc_no_imp, 3),
    "baseline_flags_total": int(baseline_flag.sum()),
    "bulk_event_rows_pct": round(df['bulk_event_flag'].mean() * 100, 1),
}
with open("work/outputs/playbook_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("Capstone metrics match deployed paper:", metrics)

Capstone metrics match deployed paper: {'honest_auc_client_grouped': np.float64(0.605), 'naive_auc_ungrouped_DO_NOT_QUOTE': np.float64(0.745), 'auc_without_impressions_90d': np.float64(0.551), 'baseline_flags_total': 10, 'bulk_event_rows_pct': np.float64(55.8)}


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.